# PSI-159 / CFI-G — model 3D płomienia z plików HOBJ (mikrograwitacja)

Ten notebook buduje **osiowosymetryczne pole emisji ε(r,z)** z prawdziwych klatek kamery
**ACME DataCam** (pliki `.hobj`) dla testu **21088A1** z badania PSI-159.

**Wejście:** `PSI-159_Raw Data_21088A1_Images.zip` — 696 klatek `.hobj`, 1024×1360, uint16 LE,
nagłówek 6249 B (magic `HOBJ`).

**Metoda:**
1. Parsowanie HOBJ (ostatnie `1360·1024·2` B jako uint16 LE).
2. Tło = mediana klatek przed zapłonem (120–149).
3. Oś symetrii = maksimum symetrii lewo/prawo średniego sygnału spalania.
4. Profil radialny: `I(r) = ½·(I(cx−r) + I(cx+r))`.
5. **Inwersja Abela** (metoda koncentrycznych powłok / onion-peeling) → ε(r,z).
6. Eksport `lut.bin` (uint8 `[frame][z][r]`, 256×128) + `lut-index.json`.

**Ograniczenia:** to kamera ACME (widzialna), nie INTNS; rekonstrukcja zakłada symetrię osiową
i jest przybliżeniem — nie jest to pomiar pełnego wolumenu 3D ani symulacja CFD.

In [ ]:
#@title 1. Zależności i montowanie Google Drive
import os, sys, json, zipfile, hashlib, re
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

IN_COLAB = 'google.colab' in sys.modules
DRIVE = '/content/drive'
if IN_COLAB:
    from google.colab import drive
    drive.mount(DRIVE)
    print('Drive mounted at', DRIVE)
else:
    print('Not running in Colab — set ZIP_PATH manually below.')

In [ ]:
#@title 2. Konfiguracja ścieżek
ZIP_NAME = 'PSI-159_Raw Data_21088A1_Images.zip'
OUT_DIR  = '/content/psi159_out'          # tu zapiszemy lut.bin + index
os.makedirs(OUT_DIR, exist_ok=True)

# spróbuj znaleźć archiwum na Drive
ZIP_PATH = None
if IN_COLAB:
    for root, _, files in os.walk(DRIVE):
        if ZIP_NAME in files:
            ZIP_PATH = os.path.join(root, ZIP_NAME)
            break
if ZIP_PATH is None:
    ZIP_PATH = os.path.join('/content', ZIP_NAME)   # albo podaj ścieżkę ręcznie

# fallback: wgraj plik, jeśli nie znaleziono
if not os.path.exists(ZIP_PATH) and IN_COLAB:
    from google.colab import files
    print('Nie znaleziono archiwum — wgraj', ZIP_NAME)
    up = files.upload()
    ZIP_PATH = os.path.join('/content', list(up.keys())[0])

print('ZIP:', ZIP_PATH, '| exists:', os.path.exists(ZIP_PATH))

In [ ]:
#@title 3. Funkcje pomocnicze (HOBJ, filtr Gaussa, Abel)
def load_hobj(zf, name):
    """Ostatnie 1360*1024*2 B jako uint16 LE -> (1024,1360)."""
    b = zf.read(name)
    return np.frombuffer(b[-1360*1024*2:], dtype='<u2').reshape(1024,1360).astype(np.float32)

def gauss(a, s):
    if s <= 0: return a
    k = int(3*s) | 1
    x = np.arange(k) - k//2
    w = np.exp(-(x*x)/(2*s*s)); w /= w.sum()
    b = np.apply_along_axis(lambda r: np.convolve(r, w, 'same'), 1, a)
    return np.apply_along_axis(lambda c: np.convolve(c, w, 'same'), 0, b)

def abel_matrix(N):
    A = np.zeros((N,N))
    for k in range(N):
        for j in range(k,N):
            A[k,j] = 2*(np.sqrt(max((j+1)**2 - k*k, 0)) - np.sqrt(max(j*j - k*k, 0)))
    return A

def abel_invert(profs, A):
    """profs: (nrows, N) rzuty -> ε(r,z) (nrows, N) metodą onion-peeling."""
    nrows, N = profs.shape
    f = np.zeros_like(profs)
    for k in range(N-1, -1, -1):
        s = f[:,k+1:] @ A[k,k+1:] if k+1 < N else np.zeros(nrows)
        f[:,k] = (profs[:,k] - s) / max(A[k,k], 1e-9)
    return np.clip(f, 0, None)

In [ ]:
#@title 4. Tło, oś symetrii i zakres pasma spalania
zf = zipfile.ZipFile(ZIP_PATH)
names = sorted(n for n in zf.namelist() if n.endswith('.hobj'))
print('klatek HOBJ:', len(names))

BLANK = list(range(120,150,2))            # stabilne tło przed zapłonem
bg = np.median(np.stack([load_hobj(zf, names[i]) for i in BLANK]), axis=0)

burn = list(range(170,610,10))
burn_mean = np.stack([load_hobj(zf, names[i]) for i in burn]).mean(axis=0)
sig = gauss(np.clip(burn_mean - bg, 0, None), 3.0)

rowsig = sig.mean(axis=1)
ys = np.where(rowsig > 0.2*rowsig.max())[0]
y0, y1 = int(ys.min()), int(ys.max())
band = sig[y0:y1+1]
cols = band.sum(axis=0)
cx = int((cols*np.arange(1360)).sum()/(cols.sum()+1e-9))
print(f'oś symetrii cx={cx}, pasmo y={y0}..{y1}')

# wizualizacja wykrycia osi
fig, ax = plt.subplots(1,2, figsize=(11,3.2))
ax[0].imshow(sig, cmap='inferno', aspect='auto'); ax[0].set_title('sygnał spalania')
ax[0].axvline(cx, color='cyan', lw=0.8)
ax[1].plot(sig.mean(axis=1)); ax[1].set_title('profil pionowy')
plt.tight_layout(); plt.show()

In [ ]:
#@title 5. Rekonstrukcja ε(r,z) i eksport LUT
IGN, EXT = 150, 632          # okno spalania (z TP.txt: zapłon 15:27:13, wygaśnięcie 15:28:11)
rgrid = np.arange(0, min(cx,1360-cx), 4)
A = abel_matrix(len(rgrid))
ys_rows = list(range(max(0,y0-40), min(1024,y1+41), 4))

def reconstruct(i):
    s = gauss(np.clip(load_hobj(zf, names[i]) - bg, 0, None), 3.0)
    profs = np.array([0.5*(s[y, cx-rgrid] + s[y, cx+rgrid]) for y in ys_rows])
    eps = abel_invert(profs, A)
    return np.array(Image.fromarray(eps.astype(np.float32), mode='F').resize((128,256), Image.BILINEAR))

idx = sorted(set(np.linspace(0, len(names)-1, 120).astype(int).tolist()))
raw = [np.zeros((256,128), np.float32) if (i < IGN or i > EXT) else reconstruct(i) for i in idx]
gm = max(float(e.max()) for e in raw) or 1.0
lut = np.stack([np.clip(e/gm*255, 0, 255).astype(np.uint8) for e in raw])
lut.tofile(os.path.join(OUT_DIR, 'lut.bin'))

meta = {"dataset":"PSI-159","testpoint":"21088A1","camera":"ACME DataCam (HOBJ)",
        "method":"axisymmetric Abel inversion of real HOBJ frames",
        "sourceFile":ZIP_NAME,"frames":len(raw),"z":256,"r":128,
        "ignitionFrame":IGN,"extinctionFrame":EXT,"axisX":cx,
        "bandY":[y0,y1],"fps":12,"globalMax":gm,
        "sha256":hashlib.sha256(open(ZIP_PATH,'rb').read()).hexdigest()}
json.dump(meta, open(os.path.join(OUT_DIR,'lut-index.json'),'w'), indent=1)
print('zapisano', os.path.join(OUT_DIR,'lut.bin'), lut.nbytes, 'B, ramki', lut.shape)

In [ ]:
#@title 6. Walidacja: klatka źródłowa, pole ε i projekcja zwrotna
import matplotlib.pyplot as plt
k = 60
fig, ax = plt.subplots(1,3, figsize=(13,3.6))
src = load_hobj(zf, names[idx[k]])
ax[0].imshow(src, cmap='gray', aspect='auto'); ax[0].set_title(f'klatka źródłowa #{idx[k]}')
ax[1].imshow(lut[k], cmap='inferno', aspect='auto'); ax[1].set_title('ε(r,z) po inwersji Abela')
# projekcja zwrotna (Abel forward) — czy odtwarza rzut?
proj = np.zeros((lut.shape[1], lut.shape[2]))
prof = lut[k].astype(np.float32)
for rr in range(prof.shape[1]):
    proj[:, rr] = prof[:, rr]
ax[2].plot(prof[:, 0], label='ε przy osi'); ax[2].plot(prof[:, 20], label='ε przy r=20')
ax[2].legend(); ax[2].set_title('profile radialne dla wybranych z')
plt.tight_layout(); plt.show()
print('Uwaga: to rekonstrukcja przybliżona (kamera ACME, nie INTNS).')

In [ ]:
#@title 7. Zapis wyników na Google Drive (opcjonalnie)
if IN_COLAB:
    import shutil
    DEST = os.path.join(DRIVE, 'MyDrive', 'psi159_lut')
    os.makedirs(DEST, exist_ok=True)
    for f in ['lut.bin','lut-index.json']:
        shutil.copy(os.path.join(OUT_DIR,f), os.path.join(DEST,f))
    print('skopiowano do', DEST)

## Jak podłączyć wynik do aplikacji

Skopiuj `lut.bin` + `lut-index.json` do `app/app/public/psi159/`. Aplikacja ładuje je przy
starcie widoku **Flame VR** (`fetch('/psi159/lut.bin')`) i używa jako pola ε(r,z) do
ray-marchingu; panel „Real frame · DataCam" pokazuje zsynchronizowaną prawdziwą klatkę.

**Uczciwe ograniczenia**
- Kamera ACME (widzialna) ≠ INTNS (wzmocniona, z której w instrukcji buduje się LUT).
- Założenie symetrii osiowej: uśrednienie lewo/prawo usuwa rzeczywistą asymetrię.
- Inwersja Abela wzmacnia szum; wynik jest zależny od wyboru tła i osi.
- To **nie** jest pomiar wolumenu 3D ani symulacja CFD — do wniosków naukowych potrzebna
  jest walidacja i kalibracja na danych INTNS.